# P1.2 Drain3 Template Analysis

Goal: Parse LANL auth logs, inspect top templates, and save Drain3 state.

In [7]:
import sys
from collections import Counter
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# Resolve repo root robustly, regardless of notebook kernel working directory.
candidates = [Path.cwd(), Path.cwd().parent, Path('.').resolve(), Path('..').resolve()]
project_root = next(
    (p.resolve() for p in candidates if (p / 'src').exists() and (p / 'data').exists()),
    None,
 )
if project_root is None:
    raise FileNotFoundError("Could not locate project root containing 'src' and 'data'.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.parsing.drain3_parser import LogParser

DATA_PATH = project_root / 'data' / 'lanl' / 'auth.txt'
MAX_LINES = 100_000   # start with 100_000, later move to 1_000_000

if not DATA_PATH.exists():
    raise FileNotFoundError(f"LANL auth log file not found at: {DATA_PATH}")

parser = LogParser(
    config_path=str(project_root / 'configs' / 'drain3.ini'),
    state_path=str(project_root / 'data' / 'drain3_state.bin')
)
template_counter = Counter()
example_rows = []

In [8]:
with DATA_PATH.open("r", encoding="utf-8", errors="ignore") as f:
    for idx, line in enumerate(tqdm(f, total=MAX_LINES, desc="Parsing LANL auth logs")):
        if idx >= MAX_LINES:
            break

        clean_line = line.strip()
        if not clean_line:
            continue

        template_id, params = parser.parse(clean_line)
        template_counter[template_id] += 1

        if len(example_rows) < 5:
            example_rows.append({
                "line": clean_line,
                "template_id": template_id,
                "params": params
            })

top20 = template_counter.most_common(20)
df_top = pd.DataFrame(top20, columns=["template_id", "count"])
display(df_top)
display(pd.DataFrame(example_rows))

print("Unique templates:", parser.get_template_count())
parser.save_state()

Parsing LANL auth logs:   0%|          | 0/100000 [00:00<?, ?it/s]

Parsing LANL auth logs: 100%|██████████| 100000/100000 [13:05<00:00, 127.24it/s]


,template_id,count
0,841,567
1,2309,96
2,4801,83
3,3074,81
4,2308,76
5,28550,36
6,3351,31
7,27698,31
8,27210,30
9,47528,25


,line,template_id,params
0,"1,ANONYMOUS LOGON@C586,ANONYMOUS LOGON@C586,C1...",1,[]
1,"1,ANONYMOUS LOGON@C586,ANONYMOUS LOGON@C586,C5...",1,"[LOGON@C586,C586,C586,?,Network,LogOff,Success]"
2,"1,C101$@DOM1,C101$@DOM1,C988,C988,?,Network,Lo...",2,[]
3,"1,C1020$@DOM1,SYSTEM@C1020,C1020,C1020,Negotia...",3,[]
4,"1,C1021$@DOM1,C1021$@DOM1,C1021,C625,Kerberos,...",4,[]


Unique templates: 5000


In [9]:
# Fill in the blanks: replace __1__ to __5__
cluster_map = parser.miner.drain.id_to_cluster
df_top["template"] = df_top["template_id"].apply(
    lambda tid: cluster_map[tid].get_template() if tid in cluster_map else "<MISSING>"
)
display(df_top[["template_id", "count", "template"]].head(20))

template_ratio = parser.get_template_count() / MAX_LINES
print("Template ratio:", round(template_ratio, 4))
if template_ratio > 0.10:
    print("Too many templates: consider increasing sim_th (e.g., 0.45-0.55).")
else:
    print("Template count is within a reasonable early-run range.")

,template_id,count,template
0,841,567,"<*> LOGON@C586,ANONYMOUS LOGON@C586,C586,C586,..."
1,2309,96,"<*> LOGON@C457,ANONYMOUS LOGON@C457,C457,C457,..."
2,4801,83,"<*> LOGON@C467,ANONYMOUS LOGON@C467,C467,C467,..."
3,3074,81,"<*> LOGON@C529,ANONYMOUS LOGON@C529,C529,C529,..."
4,2308,76,"<*> LOGON@C1909,ANONYMOUS LOGON@C1909,C1909,C1..."
5,28550,36,<MISSING>
6,3351,31,<MISSING>
7,27698,31,<MISSING>
8,27210,30,<MISSING>
9,47528,25,<MISSING>


Template ratio: 0.05
Template count is within a reasonable early-run range.


## Exercise Pack: Remaining Phase 1 (P1.3 to P1.5)

Fill all blanks like __1__, __2__, etc.
Do not run blank cells until you complete them.

After you fill a cell, send your answers and I will check before revealing the final solution.

In [10]:
# P1.3 Exercise A: Privacy hashing + Session dataclass
from dataclasses import dataclass
from datetime import datetime
import hashlib

def short_hash(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()[:8]

@dataclass
class Session:
    user_id: str
    host_id: str
    start_ts: datetime
    end_ts: datetime
    events: list[dict]
    label: int | None = None

# sanity check
print(short_hash("U123"), len(short_hash("U123")))

64a7152b 8


In [11]:
# P1.3 Exercise B: Sliding window session builder logic
from datetime import timedelta

def build_windows(events, window_mins=30, stride_mins=15, min_events=3):
    """
    events: list[dict] sorted by ts, where each event has at least a 'ts' key.
    """
    sessions = []
    if not events:
        return sessions

    start = events[0]["ts"]
    end_limit = events[-1]["ts"]
    window = timedelta(minutes=window_mins)
    stride = timedelta(minutes=stride_mins)

    while start <= end_limit:
        end = start + window
        bucket = [e for e in events if start <= e["ts"] < end]
        if len(bucket) >= min_events:
            sessions.append({
                "start_ts": start,
                "end_ts": end,
                "events": bucket[-512:]  # keep most recent 512
            })
        start = start + stride

    return sessions

In [12]:
# P1.4 Exercise A: Vocabulary builder
SPECIAL = {"[CLS]": 0, "[SEP]": 1, "[MASK]": 2, "[PAD]": 3, "[UNK]": 4}

def token_key(event: dict) -> str:
    return "|".join([
        str(event.get("EventID", "NA")),
        str(event.get("auth_type", "NA")),
        str(event.get("logon_type", "NA"))
    ])

def build_vocab(train_sessions, min_freq=5):
    from collections import Counter
    counts = Counter()
    for session in train_sessions:
        for event in session["events"]:
            counts[token_key(event)] += 1

    vocab = dict(SPECIAL)
    next_id = len(vocab)

    for key, count in counts.items():
        if count >= min_freq:
            vocab[key] = next_id
            next_id += 1

    return vocab

In [13]:
# P1.4 Exercise B: Tokenizer behavior
def tokenize_session(session, vocab, max_len=512):
    cls_id = vocab["[CLS]"]
    sep_id = vocab["[SEP]"]
    unk_id = vocab["[UNK]"]
    pad_id = vocab["[PAD]"]

    ids = [cls_id]
    for event in session["events"]:
        key = token_key(event)
        ids.append(vocab.get(key, unk_id))
    ids.append(sep_id)

    if len(ids) > max_len:
        ids = ids[-max_len:]
        ids[0] = cls_id  # keep CLS at first position after truncation

    if len(ids) < max_len:
        ids.extend([pad_id] * (max_len - len(ids)))

    return ids

In [14]:
# P1.5 Exercise: Unit-test style quick checks
def quick_checks(sample_session, vocab):
    ids = tokenize_session(sample_session, vocab, max_len=32)
    assert ids[0] == vocab["[CLS]"]
    assert vocab["[SEP]"] in ids
    assert len(ids) == 32
    return "checks passed"